# Mobile App Analytics

**Author:** Jelena Kosenko  
**GitHub:** [jelena-kosenko/data-analytics-portfolio](https://github.com/jelena-kosenko/data-analytics-portfolio)  
**Tools:** Python (pandas, numpy, matplotlib, sqlite3), Looker Studio

---

## Business Problem

> *"Users are dropping off after the first week. We don't understand why. Figure it out."*

A product manager comes to the analyst with a retention problem. This project simulates a real-world mobile app analysis: from raw data to actionable business recommendations.

## Goals
- Build and clean a user dataset
- Generate a realistic event log (funnel)
- Answer key product questions using SQL
- Identify where and why users drop off
- Compare free vs premium user behaviour

## Dataset
Synthetic dataset of **4,758 users** registered January–July 2024, with **13,657 events** across 4 event types.

---

## Step 1 — Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

np.random.seed(42)  # fix random seed for reproducibility
n = 5000            # number of users

# --- generate users table ---
users = pd.DataFrame({

    # unique user ID: 1 to 5000
    'user_id': range(1, n + 1),

    # registration date: hourly timestamps over 6 months
    'registration_date': pd.date_range('2024-01-01', periods=n, freq='h')[:n],

    # platform: 60% iOS, 40% Android
    'platform': np.random.choice(['iOS', 'Android'], n, p=[0.6, 0.4]),

    # country: weighted distribution across key markets
    'country': np.random.choice(
        ['US', 'UK', 'DE', 'BR', 'IN'], n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),

    # user type: freemium model — 75% free, 25% premium
    'user_type': np.random.choice(
        ['free', 'premium'], n, p=[0.75, 0.25]
    ),

    # age: normal distribution, clipped to realistic range 18-65
    'age': np.clip(np.random.normal(32, 10, n), 18, 65).astype(int),
})

# --- inject data quality issues (simulating real-world data) ---

# 3% duplicate rows
duplicates = users.sample(frac=0.03, random_state=1)
users = pd.concat([users, duplicates]).reset_index(drop=True)

# 5% missing age values (user did not provide)
users.loc[users.sample(frac=0.05).index, 'age'] = np.nan

# impossible age values (data entry errors)
users.loc[10, 'age'] = 200
users.loc[20, 'age'] = -5

print('Dataset shape:', users.shape)
print('\nFirst 5 rows:')
print(users.head())
print('\nData types:')
print(users.dtypes)

## Step 2 — Data Cleaning

Real-world data is never clean. We handle:
- **Duplicates** — same row appearing multiple times
- **Missing values** — NaN in the age column
- **Outliers** — impossible age values (-5, 200)
- **Data types** — converting float to int after cleaning

In [ ]:
print('=' * 60)
print('STEP 2: DATA CLEANING')
print('=' * 60)

# --- initial state ---
print(f'\n1. Initial rows: {len(users)}')
dup_count = len(users) - len(users.drop_duplicates())
print(f'   Duplicates found: {dup_count}')
print(f'   Missing values per column:')
for col in users.columns:
    n_null = users[col].isnull().sum()
    if n_null > 0:
        print(f'   {col}: {n_null}')

# --- remove duplicates ---
print(f'\n2. Removing duplicates...')
before = len(users)
users = users.drop_duplicates(keep='first')
print(f'   Removed: {before - len(users)} rows | Remaining: {len(users)}')

# --- handle missing values ---
print(f'\n3. Handling missing values...')
age_nulls = users['age'].isnull().sum()
age_pct = (age_nulls / len(users)) * 100
print(f'   Missing age: {age_nulls} rows ({age_pct:.1f}%)')
before = len(users)
users = users.dropna(subset=['age'])
print(f'   Removed: {before - len(users)} rows | Remaining: {len(users)}')

# --- remove outliers ---
print(f'\n4. Checking outliers in age...')
print(f'   Min: {users["age"].min()} | Max: {users["age"].max()}')
before = len(users)
users = users[(users['age'] >= 18) & (users['age'] <= 65)]
print(f'   Removed {before - len(users)} outlier rows | Remaining: {len(users)}')

# --- fix data types ---
print(f'\n5. Converting age: {users["age"].dtype} -> ', end='')
users['age'] = users['age'].astype(int)
print(users['age'].dtype)

# --- summary ---
print(f'\n✅ Clean dataset: {len(users)} rows, {len(users.columns)} columns')
print(f'   Total missing values: {users.isnull().sum().sum()}')

## Step 3 — Event Log Generation

Every action a user takes is tracked as an **event**. We simulate a 4-step funnel:

| Event | Description | % of users |
|---|---|---|
| `open` | User opens the app | 100% |
| `onboarding_done` | User completes onboarding | 70% |
| `purchase` | User makes first purchase | 17% |
| `close` | User closes the app | 100% |

Premium users have 3× higher purchase probability (45% vs 15%).

In [ ]:
events_list = []

for user_id in users['user_id'].values:
    reg_date = users[users['user_id'] == user_id]['registration_date'].iloc[0]
    user_type = users[users['user_id'] == user_id]['user_type'].iloc[0]

    # all users open the app
    events_list.append({'user_id': user_id, 'event': 'open',
        'date': reg_date + pd.Timedelta(hours=1)})

    # 70% complete onboarding
    if np.random.random() < 0.70:
        events_list.append({'user_id': user_id, 'event': 'onboarding_done',
            'date': reg_date + pd.Timedelta(hours=2)})

        # premium users purchase more often (45% vs 15%)
        buy_prob = 0.45 if user_type == 'premium' else 0.15
        if np.random.random() < buy_prob:
            events_list.append({'user_id': user_id, 'event': 'purchase',
                'date': reg_date + pd.Timedelta(days=int(np.random.randint(1, 15)))})

    # all users eventually close the app (1-29 days after registration)
    events_list.append({'user_id': user_id, 'event': 'close',
        'date': reg_date + pd.Timedelta(days=int(np.random.randint(1, 30)))})

events = pd.DataFrame(events_list)
events = events.sort_values(['user_id', 'date']).reset_index(drop=True)

print(f'Total events: {len(events)}')
print(f'Unique users: {events["user_id"].nunique()}')
print('\nFunnel breakdown:')
for event in ['open', 'onboarding_done', 'purchase', 'close']:
    count = (events['event'] == event).sum()
    pct = count / events['user_id'].nunique() * 100
    print(f'  {event}: {count} ({pct:.0f}%)')

print('\nSample events:')
print(events.head(15))

## Step 4 — SQL Analysis

We load both tables into an in-memory SQLite database and run three analytical queries:

1. **Funnel** — how many users complete each step?
2. **Segmentation** — how do free vs premium users differ?
3. **Retention** — how many users are still active after 7 days?

In [ ]:
# load tables into in-memory SQLite database
conn = sqlite3.connect(':memory:')
users.to_sql('users', conn, if_exists='replace', index=False)
events.to_sql('events', conn, if_exists='replace', index=False)
print(f'✅ Loaded into database: users ({len(users)} rows), events ({len(events)} rows)')

In [ ]:
# --- Query 1: Conversion Funnel ---
query_funnel = """
SELECT
    event,
    COUNT(DISTINCT user_id) AS users_count,
    ROUND(COUNT(DISTINCT user_id) * 100.0 / (
        SELECT COUNT(DISTINCT user_id) FROM events WHERE event = 'open'
    ), 1) AS pct_from_top
FROM events
GROUP BY event
ORDER BY users_count DESC
"""
funnel = pd.read_sql(query_funnel, conn)
print('CONVERSION FUNNEL:')
print(funnel)

In [ ]:
# --- Query 2: Free vs Premium Segmentation ---
query_segments = """
SELECT
    u.user_type,
    COUNT(DISTINCT u.user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN e.event = 'purchase' THEN e.user_id END) AS buyers,
    ROUND(
        COUNT(DISTINCT CASE WHEN e.event = 'purchase' THEN e.user_id END) * 100.0
        / COUNT(DISTINCT u.user_id), 1
    ) AS conversion_pct
FROM users u
LEFT JOIN events e ON u.user_id = e.user_id
GROUP BY u.user_type
"""
segments = pd.read_sql(query_segments, conn)
print('FREE vs PREMIUM CONVERSION:')
print(segments)

In [ ]:
# --- Query 3: 7-Day Retention ---
query_retention = """
WITH user_lifetime AS (
    SELECT
        u.user_id,
        u.registration_date,
        MAX(e.date) AS last_event_date
    FROM users u
    LEFT JOIN events e ON u.user_id = e.user_id
    GROUP BY u.user_id, u.registration_date
)
SELECT
    COUNT(*) AS total_users,
    SUM(CASE
        WHEN julianday(last_event_date) - julianday(registration_date) >= 7
        THEN 1 ELSE 0
    END) AS active_7days,
    ROUND(SUM(CASE
        WHEN julianday(last_event_date) - julianday(registration_date) >= 7
        THEN 1 ELSE 0
    END) * 100.0 / COUNT(*), 1) AS retention_7day_pct
FROM user_lifetime
"""
retention = pd.read_sql(query_retention, conn)
print('7-DAY RETENTION:')
print(retention)

## Step 5 — Exploratory Data Analysis (EDA)

Quick overview of key distributions in the user base.

In [ ]:
print('PLATFORM SPLIT:')
print(users['platform'].value_counts())
print(users['platform'].value_counts(normalize=True).round(2))

print('\nTOP COUNTRIES:')
print(users['country'].value_counts())

print('\nFREE vs PREMIUM:')
print(users['user_type'].value_counts())
print(users['user_type'].value_counts(normalize=True).round(2))

print('\nAGE STATS:')
print(f'Mean: {users["age"].mean():.1f} | Median: {users["age"].median():.1f}')
print(f'Min: {users["age"].min()} | Max: {users["age"].max()}')

print('\nREGISTRATIONS BY MONTH:')
users['month'] = users['registration_date'].dt.month
print(users['month'].value_counts().sort_index())

## Step 6 — Visualisation

Four charts summarising the key findings.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Case 24 — Mobile App Analytics', fontsize=14)

# Funnel
axes[0,0].barh(['open', 'onboarding', 'purchase'], [4758, 3343, 798])
axes[0,0].set_title('Conversion Funnel')

# Platform split
users['platform'].value_counts().plot.pie(ax=axes[0,1], autopct='%1.0f%%')
axes[0,1].set_title('Platform Split')
axes[0,1].set_ylabel('')

# Free vs premium conversion
axes[1,0].bar(['free', 'premium'], [11.2, 32.5])
axes[1,0].set_title('Purchase Conversion Rate (%)')

# Registrations by month
users['month'].value_counts().sort_index().plot(ax=axes[1,1], marker='o')
axes[1,1].set_title('Registrations by Month')

plt.tight_layout()
plt.savefig('case24_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Business Conclusions

### Key Metrics

| Metric | Value |
|---|---|
| Onboarding completion rate | 70% |
| Overall purchase conversion | 17% |
| 7-day retention | 81.5% |
| Premium purchase conversion | 32.5% |
| Free purchase conversion | 11.2% |

### Findings & Recommendations

**1. Onboarding is the biggest drop-off point**  
30% of users leave immediately after opening the app without completing onboarding.  
→ *Recommendation: simplify the first screen, reduce onboarding steps.*

**2. Premium users convert 3× better (32.5% vs 11.2%)**  
Premium users generate disproportionately more revenue relative to their share (26% of users, ~50% of buyers).  
→ *Recommendation: introduce a premium upsell offer earlier — ideally during onboarding.*

**3. Retention is strong (81.5% at Day 7)**  
The problem is not retention — users who complete onboarding tend to stay.  
→ *The core issue is onboarding conversion, not engagement.*

**4. iOS is the dominant platform (60%)**  
→ *Recommendation: prioritise iOS bug fixes and new features.*

**5. US is the primary market (40% of users)**  
→ *Recommendation: focus localisation and marketing spend on the US market.*

In [ ]:
# export clean data for Looker Studio dashboard
users.to_csv('users.csv', index=False)
events.to_csv('events.csv', index=False)
print('✅ Files saved: users.csv, events.csv')